<a href="https://colab.research.google.com/github/Shakiba-Fatima/Project/blob/main/mynew2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()


Saving final_fraud_dataset.csv to final_fraud_dataset.csv


In [2]:
!pip install xgboost seaborn

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from xgboost import XGBClassifier


In [4]:
df = pd.read_csv('final_fraud_dataset.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,amount,hour,day_of_week,transaction_count,avg_amount,amount_deviation,is_fraud
0,559661,21658,GLASS BEURRE DISH,8,2011-07-11 13:22:00,3.95,18061.0,United Kingdom,31.60,13,0,46,46.074130,14.474130,0
1,544838,22363,GLASS JAR MARMALADE,6,2011-02-24 10:34:00,2.95,12565.0,Saudi Arabia,17.70,10,3,9,16.213333,1.486667,0
2,563091,22424,ENAMEL BREAD BIN CREAM,1,2011-08-11 17:02:00,12.75,12994.0,United Kingdom,12.75,17,3,61,15.974098,3.224098,0
3,573028,22144,CHRISTMAS CRAFT LITTLE FRIENDS,6,2011-10-27 13:29:00,2.10,16847.0,United Kingdom,12.60,13,3,12,16.120000,3.520000,0
4,550328,22055,MINI CAKE STAND HANGING STRAWBERY,2,2011-04-17 13:18:00,1.65,17582.0,United Kingdom,3.30,13,6,29,6.043793,2.743793,0


In [5]:
train, test = train_test_split(df, test_size=0.2, random_state=42)

train.to_csv('transactions_train.csv', index=False)
test.to_csv('transactions_test.csv', index=False)


In [6]:
df_train = pd.read_csv('transactions_train.csv')
df_test = pd.read_csv('transactions_test.csv')

In [7]:
drop_cols = ['transaction_id', 'customer_id', 'merchant_id']

df_train.drop(columns=drop_cols, inplace=True, errors='ignore')
df_test.drop(columns=drop_cols, inplace=True, errors='ignore')


In [9]:
print(df.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'amount', 'hour', 'day_of_week',
       'transaction_count', 'avg_amount', 'amount_deviation', 'is_fraud'],
      dtype='object')


In [11]:
from sklearn.preprocessing import LabelEncoder

le_dict = {}

categorical_cols = df_train.select_dtypes(include='object').columns

for col in categorical_cols:
    le = LabelEncoder()

    # Combine train + test
    combined = pd.concat([df_train[col], df_test[col]], axis=0).astype(str)

    le.fit(combined)

    df_train[col] = le.transform(df_train[col].astype(str))
    df_test[col] = le.transform(df_test[col].astype(str))

    le_dict[col] = le


In [12]:
X_train = df_train.drop(['is_fraud', 'InvoiceDate'], axis=1)
y_train = df_train['is_fraud']

X_test = df_test.drop(['is_fraud', 'InvoiceDate'], axis=1)
y_test = df_test['is_fraud']


In [16]:
print(X_train.dtypes)


InvoiceNo              int64
StockCode              int64
Description            int64
Quantity               int64
UnitPrice            float64
CustomerID           float64
Country                int64
amount               float64
hour                   int64
day_of_week            int64
transaction_count      int64
avg_amount           float64
amount_deviation     float64
dtype: object


In [17]:
from sklearn.preprocessing import LabelEncoder

for col in X_train.columns:
    if X_train[col].dtype == 'object':
        le = LabelEncoder()

        combined = pd.concat([X_train[col], X_test[col]], axis=0).astype(str)
        le.fit(combined)

        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))


In [18]:
print(X_train.dtypes)


InvoiceNo              int64
StockCode              int64
Description            int64
Quantity               int64
UnitPrice            float64
CustomerID           float64
Country                int64
amount               float64
hour                   int64
day_of_week            int64
transaction_count      int64
avg_amount           float64
amount_deviation     float64
dtype: object


In [22]:
for col in X_train.columns:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# Fill any NaN values created
X_train.fillna(0, inplace=True)
X_test.fillna(0, inplace=True)


In [23]:
print(X_train.dtypes)


InvoiceNo              int64
StockCode              int64
Description            int64
Quantity               int64
UnitPrice            float64
CustomerID           float64
Country                int64
amount               float64
hour                   int64
day_of_week            int64
transaction_count      int64
avg_amount           float64
amount_deviation     float64
dtype: object


In [24]:
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])


In [25]:
from xgboost import XGBClassifier

model = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:11:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [30]:
y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.05
y_pred = (y_prob >= threshold).astype(int)


In [31]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


[[13433    54]
 [   15     0]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13487
           1       0.00      0.00      0.00        15

    accuracy                           0.99     13502
   macro avg       0.50      0.50      0.50     13502
weighted avg       1.00      0.99      1.00     13502

ROC-AUC: 0.4672969031907269


Check Fraud Distribution

In [32]:
print(y_train.value_counts())


is_fraud
0    53954
1       54
Name: count, dtype: int64


using SMOTE
which creates sythetic fraud samples to balance the dataset

In [33]:
!pip install imbalanced-learn


In [35]:
from imblearn.over_sampling import SMOTE


In [36]:
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)


In [37]:
print(y_train.value_counts())
print(y_train_smote.value_counts())


is_fraud
0    53954
1       54
Name: count, dtype: int64
is_fraud
0    53954
1    53954
Name: count, dtype: int64


XGBoost

In [38]:
from xgboost import XGBClassifier

model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss'
)

model.fit(X_train_smote, y_train_smote)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:25:16] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [39]:
y_prob = model.predict_proba(X_test)[:, 1]

threshold = 0.3
y_pred = (y_prob >= threshold).astype(int)


In [40]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))


[[13468    19]
 [   15     0]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13487
           1       0.00      0.00      0.00        15

    accuracy                           1.00     13502
   macro avg       0.50      0.50      0.50     13502
weighted avg       1.00      1.00      1.00     13502

ROC-AUC: 0.519421171004177


Random Forest

In [41]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train_smote, y_train_smote)

y_pred_rf = rf.predict(X_test)


In [42]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


[[13486     1]
 [   15     0]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     13487
           1       0.00      0.00      0.00        15

    accuracy                           1.00     13502
   macro avg       0.50      0.50      0.50     13502
weighted avg       1.00      1.00      1.00     13502



Logistic Regression

In [43]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(class_weight='balanced', max_iter=1000)

lr.fit(X_train_smote, y_train_smote)

y_pred_lr = lr.predict(X_test)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [44]:
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))


[[12829   658]
 [   15     0]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97     13487
           1       0.00      0.00      0.00        15

    accuracy                           0.95     13502
   macro avg       0.50      0.48      0.49     13502
weighted avg       1.00      0.95      0.97     13502



SVM Support Vector Machine

In [ ]:
from sklearn.svm import SVC

svm = SVC(class_weight='balanced', probability=True)

svm.fit(X_train_smote, y_train_smote)

y_pred_svm = svm.predict(X_test)
